In [ ]:
from pathlib import Path

# os.chdir('..')
Path.cwd()

PosixPath('/home/emmanuel/Desktop/research_peft')

### Implementing the tokenizer

Things to do:
- load the tokenizer    (Done)
- load model            (Done)
- Download dataset 
- Preprocess dataset
- Tokenize dataset
- Create DataLoader
- Configure Trainer
- Train model
- Save checkpoint
- Load checkpoint
- Evaluate checkpoint

In [ ]:
from dataclasses import dataclass


@dataclass
class ModelConfig:
    model_name_or_path: str = "Qwen/Qwen2.5-1.5B"
    tokenizer_name: str = None
    trust_remote_code: bool = True

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


# loading the tokenizer
def load_tokenizer(model_cfg: ModelConfig):
    name = model_cfg.tokenizer_name or model_cfg.model_name_or_path
    tokenizer = AutoTokenizer.from_pretrained(
        name,
        trust_remote_code=model_cfg.trust_remote_code,
        padding_side="right",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

In [ ]:
from typing import Any

import torch


def load_base_model(model_cfg: ModelConfig):
    """Load base model"""

    model_kwargs: dict[str, Any] = {
        "pretrained_model_name_or_path": model_cfg.model_name_or_path,
        "trust_remote_code": model_cfg.trust_remote_code,
        "torch_dtype": torch.bfloat16,
        "device_map": "auto",
    }

    if model_cfg.cache_dir:
        model_kwargs["cache_dir"] = model_cfg.cache_dir
    if model_cfg.use_flash_attention:
        model_kwargs["attn_implementation"] = "flash_attention_2"

    return AutoModelForCausalLM.from_pretrained(**model_kwargs)

In [ ]:
@dataclass
class DataConfig:
    dataset_name: str | None = None
    dataset_config: str | None = None
    train_file: str | None = None
    val_file: str | None = None
    text_column: str = "text"
    prompt_column: str | None = "instruction"
    response_column: str | None = "output"
    max_seq_length: int = 2048
    instruction_template: str = "### Instruction:\n"
    response_template: str = "### Response:\n"
    val_split: float = 0.05
    num_proc: int = 4

Todo:
- download dataset
- process and prepare dataset
- save dataset
- tokenize dataset

In [ ]:
def download_and_process_dataset():
    pass

In [ ]:
from datasets import load_dataset

ds = load_dataset("yahma/alpaca-cleaned")
# save to /home/emmanuel/Desktop/research_peft/datasets
ds.save_to_disk("/home/emmanuel/Desktop/research_peft/dataset/alpaca-cleaned")

ImportError: cannot import name 'load_dataset' from 'datasets' (/home/emmanuel/Desktop/research_peft/datasets/__init__.py)

In [ ]:
import pandas as pd

df = pd.read_json("hf://datasets/yahma/alpaca-cleaned/alpaca_data_cleaned.json")
# save to /home/emmanuel/Desktop/research_peft/datasets

df.to_json(
    "/home/emmanuel/Desktop/research_peft/dataset/alpaca_data_cleaned.json",
    orient="records",
    indent=2,
)

In [ ]:
from scripts.trainer import load_and_prepare_data

data = "dataset/alpaca_data_cleaned.json"

data_config = DataConfig(train_file=data)
train_data, val_data = load_and_prepare_data(data_config)

Generating train split: 51760 examples [00:06, 7527.32 examples/s]
INFO:scripts.trainer:Train: 49,172 | Val: 2,588


In [ ]:
# Dataset Preparation
def formt_instruction(sample: dict, data_cfg: DataConfig) -> str:
    """Format dataset into instruction-following format."""
    instruct = []
    for item in sample:
        instruction = item.get(data_cfg.prompt_column, "")
        context = item.get("input", "")
        response = item.get(data_cfg.response_column, "")

        if context:
            instruct.append(
                f"{data_cfg.instruction_template}{instruction}\n\n"
                f"### Context:\n{context}\n\n"
                f"{data_cfg.response_template}{response}"
            )
        instruct.append(
            f"{data_cfg.instruction_template}{instruction}\n\n{data_cfg.response_template}{response}"
        )

    # ds = Dataset.from_list(instruct)
    return instruct

In [43]:
# from dataset.download import ALPACA_PROMPT
ALPACA_PROMPT = (
    # "Below is an instruction that describes a task{context_str}. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "{input_block}"
    "### Response:\n{output}"
)

for i in val_data:
    [ALPACA_PROMPT.format((row) for row in i)]
    break

KeyError: 'instruction'

In [ ]:
import re
import string


def normalize_answer(text):
    """Standard normalization for QA evaluation."""
    text = text.lower()
    text = re.sub(r"^(a|an|the)\s+", "", text)  # Remove leading articles
    text = "".join(char for char in text if char not in set(string.punctuation))

    return " ".join(text.split())  # Normalize whitespace


def compute_exact_match_score(prediction, references):
    """Returns 1.0 if normalized prediction matches any reference, else 0.0."""
    normalized_pred = normalize_answer(prediction)
    normalized_refs = [normalize_answer(ref) for ref in references]
    return float(normalized_pred in normalized_refs)

In [ ]:
import json
import logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from datasets import Dataset
from tqdm import tqdm

logger = logging.getLogger(__name__)


# Perplexity
@torch.inference_mode()
def compute_perplexity(
    model,
    tokenizer,
    texts: list[str],
    max_length: int = 1024,
    stride: int = 512,
) -> dict[str, float]:
    """
    Sliding-window perplexity (handles texts longer than context window).
    Returns mean PPL, median PPL, and standard deviation across the corpus.
    """
    model.eval()
    ppls = []

    for text in tqdm(texts, desc="Computing perplexity"):
        encodings = tokenizer(text, return_tensors="pt", truncation=False)
        input_ids = encodings.input_ids.to(model.device)
        seq_len = input_ids.size(1)

        nlls = []
        prev_end = 0

        for begin in range(0, seq_len, stride):
            end = min(begin + max_length, seq_len)
            target_len = end - prev_end
            input_chunk = input_ids[:, begin:end]
            target_chunk = input_chunk.clone()
            target_chunk[:, :-target_len] = -100  # mask previously seen tokens

            with torch.no_grad():
                outputs = model(input_chunk, labels=target_chunk)
                neg_ll = outputs.loss * target_len
            nlls.append(neg_ll)
            prev_end = end
            if end == seq_len:
                break

        ppl = torch.exp(torch.stack(nlls).sum() / prev_end).item()
        ppls.append(ppl)

    return {
        "perplexity_mean": round(float(np.mean(ppls)), 4),
        "perplexity_std": round(float(np.std(ppls)), 4),
        "perplexity_median": round(float(np.median(ppls)), 4),
    }


# Generation Quality Metrics
def compute_rouge(predictions: list[str], references: list[str]) -> dict[str, float]:
    """ROUGE-1, ROUGE-2, ROUGE-L scores."""
    try:
        from rouge_score import rouge_scorer

        scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        scores = {"rouge1": [], "rouge2": [], "rougeL": []}
        for pred, ref in zip(predictions, references):
            s = scorer.score(ref, pred)
            for k in scores:
                scores[k].append(s[k].fmeasure)
        return {k: round(float(np.mean(v)), 4) for k, v in scores.items()}

    except ImportError:
        logger.warning("Install rouge-score: pip install rouge-score")
        return {}


def compute_bertscore(
    predictions: list[str],
    references: list[str],
    lang: str = "en",
    model_type: str = "microsoft/deberta-xlarge-mnli",
) -> dict[str, float]:
    """BERTScore F1 (semantic similarity)."""
    try:
        from bert_score import score

        P, R, F1 = score(predictions, references, lang=lang, model_type=model_type, verbose=False)
        return {
            "bertscore_precision": round(P.mean().item(), 4),
            "bertscore_recall": round(R.mean().item(), 4),
            "bertscore_f1": round(F1.mean().item(), 4),
        }
    except ImportError:
        logger.warning("Install bert-score: pip install bert-score")
        return {}


def compute_bleu(prediction: list[str], reference: list[str]) -> dict[str, float]:

    import sacrebleu

    try:
        # Compute corpus BLEU
        res = []
        for pred, ref in zip(prediction, reference):
            result = sacrebleu.corpus_bleu(pred, ref)
            res.append(result.score)

        return {"Bleu Score": np.mean(res)}

    except ImportError:
        logger.warning("insall sacrebleu: pip install sacrebleu")
        return {}


# Instruction-Following Benchmark
@dataclass
class BenchmarkResult:
    model_name: str
    task: str
    metrics: dict[str, float] = field(default_factory=dict)
    num_samples: int = 0
    generation_config: dict = field(default_factory=dict)


def run_generation_benchmark(
    model,
    tokenizer,
    dataset: Dataset,
    prompt_template: str,
    reference_column: str = "output",
    max_new_tokens: int = 256,
    batch_size: int = 8,
    temperature: float = 0.1,
) -> BenchmarkResult:
    """Run full generation benchmark on a dataset split."""
    predictions, references = [], []

    for i in tqdm(range(0, len(dataset), batch_size), desc="Evaluating"):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        prompts = [prompt_template.format(**row) for row in batch]
        refs = [row[reference_column] for row in batch]

        ## encode input
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(model.device)
        # run model generation
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                pad_token_id=tokenizer.eos_token_id,
            )

        # decode the input
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Strip prompt from output
        for prompt, dec, ref in zip(prompts, decoded, refs):
            predictions.append(dec[len(prompt) :].strip())
            references.append(ref.strip())

    rouge = compute_rouge(predictions, references)
    bert = compute_bertscore(predictions, references)
    bleu = compute_bleu(predictions, references)

    return BenchmarkResult(
        model_name=getattr(model.config, "_name_or_path", "unknown"),
        task="generation",
        metrics={**rouge, **bert, **bleu},
        num_samples=len(dataset),
        generation_config={"max_new_tokens": max_new_tokens, "temperature": temperature},
    )


# ─────────────────────────────────────────────
# Comparison: Finetuned models
# ─────────────────────────────────────────────


def compare_models(
    base_model_path: str,
    adapter_path: str,
    eval_texts: list[str],
    prompt_template: str,
    output_json: str = "eval_comparison.json",
):
    """Side-by-side perplexity comparison of base vs fine-tuned model."""
    import LoraInference

    logger.info("Evaluating base model...")
    base_tokenizer = AutoTokenizer.from_pretrained(base_model_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path, torch_dtype=torch.bfloat16, device_map="auto"
    )
    base_ppl = compute_perplexity(base_model, base_tokenizer, eval_texts)
    base_benchmark_result = run_generation_benchmark(
        base_model, base_tokenizer, eval_texts, prompt_template
    )
    base_result = base_benchmark_result.metrics

    logger.info("Evaluating fine-tuned model...")
    ft = LoraInference(base_model_path, adapter_path)
    ft_ppl = compute_perplexity(ft.model, ft.tokenizer, eval_texts)
    benchmark_result = run_generation_benchmark(ft.model, ft.tokenizer, eval_texts, prompt_template)
    ft_result = benchmark_result.metrics

    results = {
        "base_model": {"path": base_model_path, **base_ppl, **base_result},
        "finetuned_model": {"path": adapter_path, **ft_ppl, **ft_result},
        "delta": {k: round(base_ppl[k] - ft_ppl[k], 4) for k in base_ppl},
    }

    with Path.open(output_json, "w") as f:
        json.dump(results, f, indent=2)
    logger.info(f"Results saved to {output_json}")
    return results